# llm_budget

> Pricing, the token profile, cost functions and the spending cap for the Claude API runs (Phase 10 and the follow-ups): `PRICES` with the date they were checked, `TOKEN_PROFILE` measured in the Phase 10 pilot, `costOf`, `estimateCost`, `maxCallCost`, and `Budget`, a monthly cap (UTC calendar month) shared by processes through a locked ledger.

In [ ]:
#| default_exp llm_budget

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import fcntl
import json
from datetime import datetime, timezone
from pathlib import Path

PRICES = {   # $ per million tokens (Haiku: prompts up to 100K tokens), from the pricing page 2026-10-12 (pilot, T7)
    'claude-haiku-5-5': dict(input=0.10, output=0.50, cacheWrite=0.125, cacheRead=0.01, checked='2026-10-12'),
    'claude-sonnet-5-5': dict(input=2.00, output=10.00, cacheWrite=2.50, cacheRead=0.10, checked='2026-10-12'),
}
TOKEN_PROFILE = {   # per call: (uncached input, cache-read input, output) tokens, measured in the pilot (T7, 2026-10-12)
    ('claude-haiku-5-5', 'forage'): (840, 1824, 433), ('claude-haiku-5-5', 'board'): (208, 1771, 901),
    ('claude-sonnet-5-5', 'forage'): (840, 1824, 433), ('claude-sonnet-5-5', 'board'): (214, 1743, 805),   # forage: unused
    ('claude-haiku-5-5', 'clue'): (369, 1548, 364), ('claude-sonnet-5-5', 'clue'): (214, 1743, 805),   # clue: Haiku from the pilot (2026-10-12); Sonnet provisional (board's)
    ('claude-haiku-5-5', 'profile'): (700, 0, 300), ('claude-sonnet-5-5', 'profile'): (700, 0, 300),   # the Phase 13 vote: provisional, to be measured in its pilot (T5)
}
USAGE = ('input_tokens', 'cache_creation_input_tokens', 'cache_read_input_tokens', 'output_tokens')


class BudgetExceeded(RuntimeError):
    pass


def costOf(usage, price):
    "Dollars for one reply's usage."
    return (usage.get('input_tokens', 0) * price['input'] + usage.get('cache_creation_input_tokens', 0) * price['cacheWrite']
            + usage.get('cache_read_input_tokens', 0) * price['cacheRead'] + usage.get('output_tokens', 0) * price['output']) / 1e6


def estimateCost(model, task, agents, turns, prices=PRICES, profile=TOKEN_PROFILE):
    "Expected dollars for `agents` x `turns` calls from the token profile."
    i, r, o = profile[(model, task)]
    p = prices[model]
    return agents * turns * (i * p['input'] + r * p['cacheRead'] + o * p['output']) / 1e6



def maxCallCost(model, task, maxTokens, prices=PRICES, profile=TOKEN_PROFILE):
    "Dollars for one call that uses its whole output allowance (uncached and cache-read input from the profile)."
    i, r, _ = profile[(model, task)]
    p = prices[model]
    return (i * p['input'] + r * p['cacheRead'] + maxTokens * p['output']) / 1e6

def thisMonth():
    "The current UTC month, 'YYYY-MM': the period the spending cap applies to."
    return datetime.now(timezone.utc).strftime('%Y-%m')


class Budget:
    """A monthly spending cap shared by processes through an append-only ledger under a file lock. Each new entry carries
    its UTC `month`; `spent` counts the current month only (undated legacy entries count as earlier months), `spentAll`
    every entry. `month` is injectable for tests."""
    def __init__(self, cap, ledger, month=thisMonth):
        self.cap, self.ledger, self.month = float(cap), Path(ledger), month

    def _total(self, month=None):
        if not self.ledger.exists():
            return 0.0
        total = 0.0
        with open(self.ledger) as f:
            fcntl.flock(f, fcntl.LOCK_SH)
            for line in f:
                if not line.strip() or (month is not None and '"month"' not in line):
                    continue
                e = json.loads(line)
                if month is None or e.get('month') == month:
                    total += e['cost']
            fcntl.flock(f, fcntl.LOCK_UN)
        return total

    def spent(self):
        "Dollars spent in the current UTC month."
        return self._total(self.month())

    def spentAll(self):
        "Dollars spent in every month, legacy entries included (for reports)."
        return self._total()

    def remaining(self):
        return self.cap - self.spent()

    def check(self, need=0.0):
        left = self.remaining()
        if left <= 0 or need > left:
            raise BudgetExceeded(f'need ${need:.4f}, ${left:.4f} left of the ${self.cap:.2f} cap for {self.month()}')

    def add(self, cost, note=''):
        self.ledger.parent.mkdir(parents=True, exist_ok=True)
        with open(self.ledger, 'a') as f:
            fcntl.flock(f, fcntl.LOCK_EX)
            f.write(json.dumps(dict(cost=float(cost), note=note, month=self.month())) + '\n')
            fcntl.flock(f, fcntl.LOCK_UN)


## Tests

In [ ]:
import tempfile
from pathlib import Path
from explRL.llm_budget import *

# a call that uses its whole output allowance: profile input plus maxTokens output
i, r, _ = TOKEN_PROFILE[('claude-sonnet-5-5', 'board')]
assert abs(maxCallCost('claude-sonnet-5-5', 'board', 8000) - (i * 2.0 + r * 0.10 + 8000 * 10.0) / 1e6) < 1e-15
assert maxCallCost('claude-sonnet-5-5', 'board', 8000) < 6 / 64                # 64 such calls in flight cost under $6
assert maxCallCost('claude-haiku-5-5', 'forage', 2000) < maxCallCost('claude-haiku-5-5', 'forage', 8000)
assert costOf(dict(output_tokens=10**6), PRICES['claude-haiku-5-5']) == 0.5
# the budget: a shared ledger, a cap, and a check for what a run still needs
tmp = Path(tempfile.mkdtemp())
b = Budget(1.0, tmp / 'spend.jsonl')
b.add(0.25, 'x'); b.add(0.25, 'y')
assert b.spent() == 0.5 and b.remaining() == 0.5 and Budget(1.0, tmp / 'spend.jsonl').spent() == 0.5
b.check(0.4)
try:
    b.check(0.6); raise AssertionError
except BudgetExceeded:
    pass

In [ ]:
# test_budget_month (Phase 20 NFR-7, Phase 21): the cap counts the current UTC month only
import json
tmp = Path(tempfile.mkdtemp()); led = tmp / 'spend.jsonl'
led.write_text('{"cost": 2.0, "note": "legacy"}\n{"cost": 1.0, "note": "legacy"}\n')   # undated, as before November 2026
now = ['2026-11']
b = Budget(5.0, led, month=lambda: now[0])
assert b.spent() == 0.0 and b.spentAll() == 3.0 and b.remaining() == 5.0   # legacy entries count as earlier months
b.add(1.5, 'a')
assert json.loads(led.read_text().splitlines()[-1]) == dict(cost=1.5, note='a', month='2026-11')
assert b.spent() == 1.5 and b.spentAll() == 4.5
now[0] = '2026-12'                                                             # a month boundary: the cap resets
assert b.spent() == 0.0 and b.remaining() == 5.0 and b.spentAll() == 4.5
b.add(4.0, 'b'); b.check(0.9)
try:
    b.check(1.1); raise AssertionError
except BudgetExceeded as e:
    assert '2026-12' in str(e)
now[0] = '2026-11'; assert b.spent() == 1.5
assert len(thisMonth()) == 7 and thisMonth()[4] == '-'


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()